In [64]:
# Importe le module OS pour interagir avec le système d'exploitation
import os
 # Change le répertoire de travail vers le dossier parent
os.chdir("../data")

%pwd


'C:\\Cours\\Programming\\AI\\Gen_AI\\End-to-End-Medical-chatbot-Generative-AI\\data'

In [65]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader  
from langchain.text_splitter import RecursiveCharacterTextSplitter  

In [66]:
# Extract data from PDF files
def load_pdf_file(data):
    loader = DirectoryLoader(data,  
                             glob="*.pdf",
                             loader_cls=PyPDFLoader)

    documents = loader.load()

    return documents

In [67]:
extracted_data = load_pdf_file(data="../data/")

In [68]:
# extracted_data

In [69]:
#Split the Data into chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [70]:
text_chunks = text_split(extracted_data)
print("Length of text Chunks", len(text_chunks))

Length of text Chunks 5860


In [71]:
!pip install -U langchain-huggingface
!pip uninstall sentence-transformers huggingface-hub langchain -y
!pip install sentence-transformers
!pip install huggingface-hub
!pip install langchain

Found existing installation: sentence-transformers 3.3.1
Uninstalling sentence-transformers-3.3.1:
  Successfully uninstalled sentence-transformers-3.3.1
Found existing installation: huggingface-hub 0.27.0
Uninstalling huggingface-hub-0.27.0:
  Successfully uninstalled huggingface-hub-0.27.0
Found existing installation: langchain 0.3.13
Uninstalling langchain-0.3.13:
  Successfully uninstalled langchain-0.3.13
  Using cached sentence_transformers-3.3.1-py3-none-any.whl.metadata (10 kB)
  Using cached huggingface_hub-0.27.0-py3-none-any.whl.metadata (13 kB)
Using cached sentence_transformers-3.3.1-py3-none-any.whl (268 kB)
Using cached huggingface_hub-0.27.0-py3-none-any.whl (450 kB)
  Using cached langchain-0.3.13-py3-none-any.whl.metadata (7.1 kB)
Using cached langchain-0.3.13-py3-none-any.whl (1.0 MB)


In [72]:
from langchain_huggingface import HuggingFaceEmbeddings

In [73]:

# Download the embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings


In [74]:
embeddings = download_hugging_face_embeddings()

In [75]:
# Génère l'embedding (vecteur numérique) pour la requête "Hello world" à l'aide du modèle d'embedding chargé.  
query_result = embeddings.embed_query("Hello world")  

# Affiche la longueur du vecteur généré, qui correspond au nombre de dimensions dans l'espace d'embedding.  
print("Length", len(query_result))  

import json
print(json.dumps(query_result, indent=4))

Length 384
[
    -0.03447727486491203,
    0.03102312609553337,
    0.006734980270266533,
    0.026108933612704277,
    -0.03936205804347992,
    -0.16030246019363403,
    0.06692394614219666,
    -0.006441438104957342,
    -0.047450482845306396,
    0.014758863486349583,
    0.07087534666061401,
    0.05552757531404495,
    0.019193356856703758,
    -0.02625126577913761,
    -0.01010954286903143,
    -0.026940442621707916,
    0.022307462990283966,
    -0.02222665585577488,
    -0.14969263970851898,
    -0.017493024468421936,
    0.007676282897591591,
    0.054352231323719025,
    0.0032544038258492947,
    0.03172588348388672,
    -0.08462139964103699,
    -0.029405992478132248,
    0.051595550030469894,
    0.048124078661203384,
    -0.003314835485070944,
    -0.05827915295958519,
    0.04196925833821297,
    0.022210702300071716,
    0.1281888633966446,
    -0.022338951006531715,
    -0.011656239628791809,
    0.06292837113142014,
    -0.03287634998559952,
    -0.09122604131698608,

In [76]:
from dotenv import load_dotenv
load_dotenv()

True

In [77]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')

In [83]:

from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os

# Initialisation de la connexion à Pinecone avec la clé API stockée dans les variables d'environnement.  
pc = Pinecone(
    api_key=PINECONE_API_KEY,
    environment="us-east-1",  # Remplacez par votre région
    timeout=180  # Spécifiez ici le délai d'attente
)  

# Nom de l'index à créer dans la base de données vectorielle Pinecone.  
index_name = "medicalbot"  

# Création d'un nouvel index dans Pinecone avec des paramètres spécifiques.  
pc.create_index(
    name=index_name,  # Nom de l'index utilisé pour identifier et stocker les vecteurs.  
    dimension=384,  # Dimension des vecteurs (à remplacer par la dimension réelle du modèle, par exemple 384).  
    metric="cosine",  # Métrique utilisée pour mesurer la similarité entre vecteurs (cosine dans ce cas).  
    spec=ServerlessSpec(  # Configuration du déploiement serverless (sans gestion explicite du serveur).  
        cloud="aws",  # Fournisseur cloud où l'index est hébergé (ici Amazon Web Services).  
        region="us-east-1"  # Région géographique pour optimiser la latence et la disponibilité.  
    ) 
)


PineconeApiException: (409)
Reason: Conflict
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'access-control-allow-origin': '*', 'vary': 'origin,access-control-request-method,access-control-request-headers', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2024-07', 'X-Cloud-Trace-Context': 'e63a14ba31a9297d147086ff6180466b', 'Date': 'Thu, 02 Jan 2025 05:29:32 GMT', 'Server': 'Google Frontend', 'Content-Length': '85', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"ALREADY_EXISTS","message":"Resource  already exists"},"status":409}


In [ ]:
!pip install --upgrade langchain
!pip install --upgrade langchain-pinecone
!pip install --upgrade pinecone-client
!pip install --upgrade urllib3


In [ ]:

# Intégrer chaque segment et insérer les embeddings dans votre index Pinecone.
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings,
)

In [ ]:
# Charger un index existant dans Pinecone

# Importation de la classe PineconeVectorStore depuis langchain_pinecone
from langchain_pinecone import PineconeVectorStore

# Intégrer chaque segment et insérer les embeddings dans l'index Pinecone
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,  # Nom de l'index existant dans Pinecone
    embedding=embeddings,   # Modèle d'embedding utilisé pour la correspondance sémantique
)
docsearch

In [ ]:
# Créer un récupérateur de documents basé sur la similarité
retriever = docsearch.as_retriever(
    search_type="similarity",       # Utiliser la recherche basée sur la similarité sémantique
    search_kwargs={"k": 3}         # Renvoyer les 3 documents les plus similaires
)

# Effectuer une recherche avec une requête textuelle
retrieved_docs = retriever.invoke("What is Acne?")  # Recherche des documents liés à l'acné


In [ ]:
retrieved_docs